# Harmonize scored-event labels of a Curry 9 (.cdt) EEG/PSG database

This notebook lets you **visualize** the scored-event configurations present in a Curry 9
database (events annotated during sleep scoring and exported by Curry as a text file:
arousals, apnea, hypopnea, limb movements, desaturations…) and **harmonize** their raw
labels to a single canonical vocabulary.

It returns a JSON file `config_param/event_remap.json` (a flat python dict
`{raw_label: canonical_label}`, with `null` for labels you choose to ignore) that downstream
tools (epoch rejection in `6_preprocessing_curry`) can use.

---
**To use this notebook, interact with the widgets and read the output below. You first have to
select a database to make the widgets appear.**

Sections:
1. Select your study folder and scan the events
2. Event configurations found
3. Harmonize the labels
4. Preview & save the JSON
5. Verify

The events are read from the Curry text export `*_ScoredEvents_Export.txt` next to each `.cdt`.
The raw labels are the export strings (often in French, e.g. `Micro-éveil 1 ARO SPONT`); the
canonical suggestions below now recognize the common French Compumedics/Curry labels (arousals,
apnea, hypopnea, desaturation, snoring…), so most are pre-filled — unusual labels still need a
manual choice.


In [ ]:
# Imports
try:
    import os
    import re
    import json
    import datetime
    import traceback
    import unicodedata
    from pathlib import Path
    from collections import Counter, OrderedDict
    import pandas as pd
    import ipywidgets as widgets
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML, clear_output
    import sys as _sys
    # curry shared modules — located next to this notebook
    _here = os.path.dirname(os.path.abspath('__file__'))
    if _here not in _sys.path:
        _sys.path.insert(0, _here)
    from curry_header import read_curry_header
    from curry_io import load_events_curry, rec_start_from_header
except ImportError as e:
    print("⚠️ Error: ", e)
else:
    print("✅ Packages and functions successfully imported!")


# ---- generic helpers (shared idioms with 2_select&remap_channels_edf) ----
def _load_json_lenient(path):
    """Load a JSON file, tolerating a single trailing comma before a closing } or ]
    (a common hand-edit mistake): strict parse first, repair only on failure."""
    with open(path, encoding="utf-8") as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return json.loads(re.sub(r",(\s*[}\]])", r"\1", text))


def print_in_scrollable_box(text, height=300, font_size="12px"):
    display(HTML(f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid black; '
                 f'padding:10px; font-size:{font_size};">{text}</pre>'))


# ---- canonical event vocabulary (editable suggestions only) ----
# Compumedics/Profusion raw labels -> harmonized snake_case canonical labels.
# Arousal subtypes are kept (clinically meaningful); laterality is dropped for limbs.
DEFAULT_EVENT_MAPPING = {
    "obstructive apnea":   "apnea_obstructive",
    "central apnea":       "apnea_central",
    "mixed apnea":         "apnea_mixed",
    "hypopnea":            "hypopnea",
    "spo2 desaturation":   "spo2_desaturation",
    "spo2 artifact":       "spo2_artifact",
    "arousal (aro res)":   "arousal_respiratory",
    "arousal (aro spont)": "arousal_spontaneous",
    "arousal (aro plm)":   "arousal_limb",
    "arousal (aro limb)":  "arousal_limb",
    "arousal ()":          "arousal",
    "arousal":             "arousal",
    "limb movement (left)":  "limb_movement",
    "limb movement (right)": "limb_movement",
    "limb movement":         "limb_movement",
    "plm (left)":  "plm",
    "plm (right)": "plm",
    "plm":         "plm",
    "snore":       "snore",
    "snoring":     "snore",
}
# ---- French text-export labels (*_ScoredEvents_Export.txt) ----
# The French export writes variable label text ("Micro-éveil 2 ARO RES"), so match on
# informative substrings (accent-insensitive) rather than exact keys.
FRENCH_EVENT_RULES = [
    ("aro spont",         "arousal_spontaneous"),
    ("aro res",           "arousal_respiratory"),
    ("aro plm",           "arousal_limb"),
    ("aro limb",          "arousal_limb"),
    ("apnee obstructive", "apnea_obstructive"),
    ("apnee centrale",    "apnea_central"),
    ("apnee mixte",       "apnea_mixed"),
    ("hypopnee",          "hypopnea"),
    ("desaturation",      "spo2_desaturation"),
    ("artefact spo2",     "spo2_artifact"),
    ("ronflement",        "snore"),
    ("plm",               "plm"),
]
CANONICAL_VOCAB = sorted(set(DEFAULT_EVENT_MAPPING.values())
                         | {canon for _, canon in FRENCH_EVENT_RULES})


def _strip_accents(text):
    """Lower-case and drop accents so French labels match regardless of accentuation
    (é→e, É→e). Used for the substring rules above."""
    return "".join(c for c in unicodedata.normalize("NFKD", text.lower())
                   if not unicodedata.combining(c))


def suggest_canonical(raw):
    """Suggest a canonical label for a raw event name (empty string if unknown).
    English labels (Compumedics CSV/XML) match an exact dict; French labels (the
    *_ScoredEvents_Export.txt export) match accent-insensitive substrings."""
    key = raw.strip().lower()
    if key in DEFAULT_EVENT_MAPPING:
        return DEFAULT_EVENT_MAPPING[key]
    # tolerate a trailing laterality marker, e.g. "Limb Movement (Left)"
    stripped = re.sub(r"\s*\((left|right)\)\s*$", "", key).strip()
    if stripped in DEFAULT_EVENT_MAPPING:
        return DEFAULT_EVENT_MAPPING[stripped]
    # French text export: informative-substring match (accent-insensitive)
    norm = _strip_accents(raw)
    for sub, canon in FRENCH_EVENT_RULES:
        if sub in norm:
            return canon
    return ""


# ---- event loading: Curry French text export (*_ScoredEvents_Export.txt) ----
def load_events(cdt_path, event_suffix="_ScoredEvents_Export.txt"):
    """Load Curry scored events next to the .cdt file (French text export).
    Returns (events_list, 'txt') with events_list = list of (name, start, duration) in
    seconds, or (None, None) when the export is absent/unreadable. The .cdt.dpo header gives
    the recording-start datetime used to convert the export clock times (load_events_curry)."""
    cdt_path = Path(cdt_path)
    ev_path = cdt_path.with_name(f"{cdt_path.stem}{event_suffix}")
    if not ev_path.exists():
        return None, None
    try:
        hdr = read_curry_header(str(cdt_path))
        rec_start = rec_start_from_header(hdr)
        if rec_start is None:
            return None, None
        df = load_events_curry(str(ev_path), rec_start)
        if df is None or df.empty:
            return None, None
        events = [(str(n).strip(), float(s), float(d))
                  for n, s, d in zip(df["Name"], df["Start"], df["Duration"])]
        return events, "txt"
    except Exception:
        return None, None


def load_existing_mapping(folder):
    """Read the existing config_param/event_remap.json (lenient), {} if absent/unreadable."""
    p = Path(folder) / "config_param" / "event_remap.json"
    if p.exists():
        try:
            return _load_json_lenient(p)
        except Exception:
            return {}
    return {}


# ---- shared state filled by the scan ----
STATE = {}


# ========================= Section banners =========================
section1 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>1. Select your study folder and scan the events</h2>
<p>Pick the folder of your Curry (.cdt) database. Each .cdt is expected to have a Curry event
text export next to it, whose suffix is set in the <b>Event export suffix</b> field below
(default <code>_ScoredEvents_Export.txt</code>).
<br>&#x2022; Selecting the folder auto-detects the suffix and refreshes the info line below.
<br>&#x2022; Click <b>Run scan</b> to read the events and list the configurations.
<br>&#x2022; "Skip labels already mapped" hides labels already present in an existing
<code>event_remap.json</code> (incremental harmonization when you add a new cohort).</p>
""")

section2 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>2. Event configurations found</h2>
<p>Files are grouped by their set of <b>unique</b> event labels. Two files with the same unique
labels share one configuration even if their event counts differ.</p>
""")

section3 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>3. Harmonize the labels</h2>
<p>For each unique raw label, choose a harmonized (canonical) label. Defaults are pre-filled
when recognized; you can edit them freely.
<br>&#x2022; Tick <b>ignore</b> to drop a label (stored as <code>null</code>, excluded downstream).
<br>&#x2022; Click <b>Validate mapping &amp; ignores</b> to check your choices and unlock section 4.
<br>&#x2022; Then go to section 4 to preview &amp; save.</p>
""")

section4 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>4. Preview &amp; save the JSON</h2>
<p>Builds a flat <code>{raw_label: canonical_label}</code> mapping and <b>merges</b> it into any
existing <code>config_param/event_remap.json</code> (labels mapped this session replace their old
value, all others are kept).</p>
""")

section5 = widgets.HTML("""
<hr style="height:4px; background-color:black; border:none;">
<h2>5. Verify</h2>
<p>Applies the saved mapping to every configuration and reports the resulting harmonized labels.
The verdict passes when no raw label is left unmapped (ignored labels count as handled).</p>
""")

# ========================= Section 1 widgets =========================
chooser = FileChooser(os.getcwd())
chooser.title = "<b>Choose your study folder</b>"
chooser.show_only_dirs = True

skip_existing = widgets.Checkbox(value=True, description="Skip labels already mapped",
                                 style={"description_width": "initial"})
existing_info = widgets.HTML(value="")
csv_suffix = widgets.Text(value="_ScoredEvents_Export.txt", description="Event export suffix:",
                          style={"description_width": "initial"},
                          layout=widgets.Layout(width="420px"))
csv_suffix_info = widgets.HTML(value="")
run_scan_button = widgets.Button(description="Run scan", button_style="success", icon="play")
out_scan = widgets.Output()

# Section 2 / 3 / 4 / 5 output zones
out_configs = widgets.Output()
out_harmonize = widgets.Output()
fname_text = widgets.Text(value="event_remap.json", description="File name:",
                          style={"description_width": "initial"}, layout=widgets.Layout(width="320px"))
preview_save_button = widgets.Button(description="Preview & Save", button_style="success",
                                     icon="save", disabled=True)
out_save = widgets.Output()
validate_button = widgets.Button(description="Validate mapping & ignores",
                                 button_style="primary", icon="check")
out_validate = widgets.Output()
verify_scope = widgets.Dropdown(options=["All configurations", "Only configs with unmapped labels"],
                                value="All configurations", description="Show:",
                                style={"description_width": "initial"}, layout=widgets.Layout(width="380px"))
verify_button = widgets.Button(description="Run verification", button_style="success", icon="play")
out_verify = widgets.Output()


# ========================= Callbacks =========================
def _update_info(*_):
    """Refresh the info line when the folder changes (no scan)."""
    if not getattr(chooser, "selected_path", None):
        existing_info.value = ""
        csv_suffix_info.value = ""
        return
    try:
        folder = Path(chooser.selected_path)
        cdts = [f for f in folder.rglob("*") if f.suffix == ".cdt" and not f.name.startswith("._")]
        if not cdts:
            existing_info.value = '<small style="color:#888;">No .cdt files found in selected folder.</small>'
            csv_suffix_info.value = ""
            return
        existing = load_existing_mapping(folder)
        msg = f"<small>{len(cdts)} .cdt file(s) found. "
        msg += (f"{len(existing)} label(s) already mapped in event_remap.json."
                if existing else "No existing event_remap.json yet.")
        existing_info.value = msg + "</small>"
        # --- Event-export suffix auto-detection (Curry: *_ScoredEvents_Export.txt) ---
        # Scan .txt files whose name contains 'event' so hypnogram .txt files are not counted.
        all_evt = [f for f in folder.rglob("*")
                   if f.suffix.lower() == ".txt" and "event" in f.name.lower()]
        suffix_counts = {}
        for cdt in cdts:
            for evtf in all_evt:
                if os.path.normcase(evtf.name).startswith(os.path.normcase(cdt.stem)):
                    suf = evtf.name[len(cdt.stem):]
                    suffix_counts[suf] = suffix_counts.get(suf, 0) + 1
        if not suffix_counts:
            csv_suffix_info.value = (
                '<small style="color:#e67e00;">No event export detected next to the .cdt files '
                '— set the suffix manually.</small>')
        else:
            # Prefer the MOST FREQUENT suffix (shortest on ties).
            best_suffix, best_count = max(
                suffix_counts.items(), key=lambda x: (x[1], -len(x[0])))
            csv_suffix.value = best_suffix
            parts = [f'<b>{s}</b>&nbsp;(×{c})'
                     for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
            color = '#2e7d32' if best_count == len(cdts) else '#e67e00'
            csv_suffix_info.value = (
                f'<small style="color:{color};">Detected:&nbsp;'
                f'{"&nbsp;·&nbsp;".join(parts)}&nbsp;— '
                f'{best_count}/{len(cdts)} .cdt file(s) matched</small>')
    except Exception as e:
        existing_info.value = f'<small style="color:#c33;">{type(e).__name__}: {e}</small>'
        csv_suffix_info.value = ""


def run_scan(_=None):
    for z in (out_scan, out_configs, out_harmonize, out_validate, out_save, out_verify):
        z.clear_output()
    with out_scan:
        if not getattr(chooser, "selected_path", None):
            print("⚠️ Please select your study folder first.")
            return
        folder = Path(chooser.selected_path)
        cdts = sorted(f for f in folder.rglob("*")
                      if f.suffix == ".cdt" and not f.name.startswith("._"))
        if not cdts:
            print("⚠️ No .cdt files found in the selected folder.")
            return
        configs = {}          # frozenset(labels) -> [file_id]
        label_files = {}      # raw label -> set(file_id)
        label_counts = {}     # raw label -> total occurrences
        source_by_file = {}   # file_id -> 'txt'/'csv'/'xml'
        failed = []           # (file_id, reason)
        n_with = 0
        for cdt in cdts:
            fid = cdt.stem
            try:
                events, source = load_events(cdt, csv_suffix.value)
            except Exception as e:
                failed.append((fid, f"{type(e).__name__}: {e}"))
                continue
            if events is None:
                failed.append((fid, "no readable event export (*_ScoredEvents_Export.txt)"))
                continue
            n_with += 1
            source_by_file[fid] = source
            names = [n for (n, _, _) in events]
            configs.setdefault(frozenset(names), []).append(fid)
            for nm in names:
                label_counts[nm] = label_counts.get(nm, 0) + 1
            for nm in set(names):
                label_files.setdefault(nm, set()).add(fid)
        STATE.update(folder=folder, configs=configs, label_files=label_files,
                     label_counts=label_counts, source_by_file=source_by_file,
                     failed=failed, existing=load_existing_mapping(folder))
        if failed:
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            pd.DataFrame(failed, columns=["file_id", "reason"]).to_csv(
                cfgdir / "failed_event_read.tsv", sep="\t", index=False)
        print(f"✅ Scanned {len(cdts)} .cdt file(s): {n_with} with events, {len(failed)} failed.")
        print(f"   {len(label_files)} unique raw label(s), {len(configs)} distinct event configuration(s).")
        if STATE["existing"]:
            print(f"   {len(STATE['existing'])} label(s) already in event_remap.json.")
        if failed:
            print(f"   ⚠ {len(failed)} file(s) without readable events "
                  f"— see config_param/failed_event_read.tsv")
    render_configs()
    render_harmonize()


def render_configs():
    with out_configs:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            return
        items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))

        # ---- per-config viewer: a dropdown selects one configuration to detail ----
        dd_options = [(f"Configuration {i} — {len(fids)} file(s), {len(labels)} unique label(s)", i - 1)
                      for i, (labels, fids) in enumerate(items, 1)]
        config_dd = widgets.Dropdown(options=dd_options, value=0, description="Show config:",
                                     style={"description_width": "initial"},
                                     layout=widgets.Layout(width="520px"))
        show_ids_btn = widgets.ToggleButton(value=False, description="Show file ids",
                                            icon="list-ul", layout=widgets.Layout(width="170px"))
        detail_out = widgets.Output()
        ids_out = widgets.Output()

        def _render_detail(*_):
            labels, fids = items[config_dd.value]
            labs = sorted(labels)
            with detail_out:
                clear_output()
                display(HTML(f"<b>{len(labs)} unique label(s)</b> in {len(fids)} file(s):<br>"
                             + "<br>".join(f"&#x2022; {l}" for l in labs)))
            show_ids_btn.description = "Hide file ids" if show_ids_btn.value else "Show file ids"
            with ids_out:
                clear_output()
                if show_ids_btn.value:
                    display(HTML('<pre style="overflow:auto; max-height:140px; border:1px solid #ccc; '
                                 'padding:6px; font-size:12px; margin-top:4px;">'
                                 + ", ".join(sorted(fids)) + "</pre>"))

        config_dd.observe(_render_detail, names="value")
        show_ids_btn.observe(_render_detail, names="value")
        display(widgets.VBox([config_dd, detail_out, show_ids_btn, ids_out]))
        _render_detail()

        # ---- global table of all unique raw labels (kept) ----
        lf, lc = STATE["label_files"], STATE["label_counts"]
        rows = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        df = pd.DataFrame([{"raw_label": l, "n_files": len(lf[l]), "n_occurrences": lc[l],
                            "suggested_canonical": suggest_canonical(l) or "(none)"} for l in rows])
        display(HTML("<h4>All unique raw labels</h4>"))
        display(HTML(df.to_html(index=False)))


def render_harmonize(*_):
    with out_harmonize:
        clear_output()
        out_validate.clear_output()
        preview_save_button.disabled = True   # require a fresh validation after any (re)build
        lf = STATE.get("label_files", {})
        if not lf:
            return
        existing = STATE.get("existing", {})
        labels = sorted(lf.keys(), key=lambda l: (-len(lf[l]), l.lower()))
        if skip_existing.value:
            labels = [l for l in labels if l not in existing]
        STATE["rows"] = {}
        row_widgets = []
        if not labels:
            display(HTML("<i>All raw labels are already mapped (uncheck “Skip labels "
                         "already mapped” to edit them).</i>"))
            return
        header = widgets.HBox([
            widgets.HTML("<b>raw label</b> (files)", layout=widgets.Layout(width="320px")),
            widgets.HTML("&rarr; <b>canonical label</b>", layout=widgets.Layout(width="240px")),
            widgets.HTML("<b>ignore</b>", layout=widgets.Layout(width="90px")),
        ])
        for raw in labels:
            if raw in existing:
                val = existing[raw]
                combo_val, ignore_val = ("", True) if val is None else (val, False)
            else:
                combo_val, ignore_val = suggest_canonical(raw), False
            lab = widgets.HTML(f"<code>{raw}</code> <small style='color:#888'>({len(lf[raw])})</small>",
                               layout=widgets.Layout(width="320px"))
            combo = widgets.Combobox(value=combo_val, options=CANONICAL_VOCAB, ensure_option=False,
                                     placeholder="canonical label", disabled=ignore_val,
                                     layout=widgets.Layout(width="240px"))
            ign = widgets.Checkbox(value=ignore_val, description="ignore", indent=False,
                                   layout=widgets.Layout(width="90px"))

            def _toggle(change, c=combo):
                c.disabled = change["new"]
            ign.observe(_toggle, names="value")
            combo.observe(_invalidate_save, names="value")
            ign.observe(_invalidate_save, names="value")
            STATE["rows"][raw] = (combo, ign)
            row_widgets.append(widgets.HBox([lab, combo, ign], layout=widgets.Layout(
                border="1px solid #e0e0e0", padding="4px", margin="0 0 2px 0",
                align_items="center", overflow="hidden")))
        box = widgets.VBox(row_widgets, layout=widgets.Layout(
            max_height="420px", overflow="auto", border="1px solid #ddd", padding="6px"))
        display(widgets.VBox([header, box]))


def _invalidate_save(*_):
    """Any change in section 3 invalidates a prior validation: the saved JSON must
    always reflect the latest selection, so re-lock section 4 until re-validated."""
    if not preview_save_button.disabled:
        preview_save_button.disabled = True
        out_save.clear_output()
        with out_validate:
            clear_output()
            display(HTML("<i style='color:#c98a00'>⚠ Selection changed — click "
                         "<b>Validate mapping &amp; ignores</b> again before saving.</i>"))


def on_validate(_=None):
    with out_validate:
        clear_output()
        rows = STATE.get("rows", {})
        if not rows:
            print("⚠️ Run the scan (section 1) first.")
            return
        mapped, ignored, empties = [], [], []
        for raw, (combo, ign) in rows.items():
            if ign.value:
                ignored.append(raw)
            elif combo.value.strip():
                mapped.append(raw)
            else:
                empties.append(raw)
        display(HTML(f"<b>Validation summary:</b> {len(mapped)} mapped, "
                     f"{len(ignored)} ignored, {len(empties)} left empty."))
        if empties:
            display(HTML("<span style='color:#c33'>⚠ empty (will NOT be saved): "
                         + ", ".join(f"<code>{e}</code>" for e in empties)
                         + " — set a canonical label or tick “ignore”.</span>"))
        display(HTML("<span style='color:#178a17'>✅ Section 4 unlocked — you can now "
                     "<b>Preview &amp; Save</b> below.</span>"))
        preview_save_button.disabled = False


def on_preview_save(_=None):
    with out_save:
        clear_output()
        rows = STATE.get("rows", {})
        if not rows:
            print("⚠️ Run the scan (section 1) first.")
            return
        try:
            session, empties = {}, []
            for raw, (combo, ign) in rows.items():
                if ign.value:
                    session[raw] = None
                else:
                    v = combo.value.strip()
                    if v:
                        session[raw] = v
                    else:
                        empties.append(raw)
            folder = STATE["folder"]
            existing = load_existing_mapping(folder)   # reload fresh to merge
            merged = dict(existing)
            merged.update(session)
            merged = OrderedDict(sorted(merged.items(), key=lambda kv: kv[0].lower()))
            cfgdir = folder / "config_param"
            cfgdir.mkdir(exist_ok=True)
            out_json = cfgdir / (fname_text.value.strip() or "event_remap.json")
            with open(out_json, "w", encoding="utf-8") as f:
                json.dump(merged, f, indent=2, ensure_ascii=False)
            n_new = len(session)
            display(HTML(f"<p><b>✅ JSON saved here:</b> <code>{out_json}</code></p>"))
            display(HTML(f"<p style='color:#555'>{len(merged)} label(s) total "
                         f"(<b>{n_new}</b> added/updated this session); previous entries preserved.</p>"))
            if empties:
                display(HTML("<p style='color:#c33'>⚠ left unmapped (not saved): "
                             + ", ".join(f"<code>{e}</code>" for e in empties)
                             + " — set a canonical label or tick “ignore”.</p>"))
            preview = json.dumps(merged, indent=2, ensure_ascii=False)
            print_in_scrollable_box(preview.replace("<", "&lt;").replace(">", "&gt;"), height=260)
            display(HTML("<br><b>To load this file later:</b>"))
            display(HTML("<p><code>with open(path, encoding='utf-8') as f: event_map = json.load(f)</code></p>"))
            display(HTML("<p><code>canonical = event_map.get(raw_label)  # None = ignore</code></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:#c33'>{type(e).__name__}: {e}\n"
                         f"{traceback.format_exc()}</pre>"))


def run_verify(_=None):
    with out_verify:
        clear_output()
        configs = STATE.get("configs", {})
        if not configs:
            print("⚠️ Run the scan (section 1) first.")
            return
        try:
            folder = STATE["folder"]
            mapping = load_existing_mapping(folder)
            if not mapping:
                print("⚠️ No event_remap.json found yet — save it in section 4 first.")
                return
            items = sorted(configs.items(), key=lambda kv: (-len(kv[1]), sorted(kv[0])))
            any_unmapped = False
            html = []
            for i, (labels, fids) in enumerate(items, 1):
                labs = sorted(labels)
                unmapped = [l for l in labs if l not in mapping]
                canon = sorted({mapping[l] for l in labs if mapping.get(l) is not None})
                if unmapped:
                    any_unmapped = True
                if verify_scope.value.startswith("Only") and not unmapped:
                    continue
                html.append(f"<h4>Configuration {i} &mdash; {len(fids)} file(s)</h4>")
                html.append("<b>Harmonized labels:</b> "
                            + (", ".join(f"<code>{c}</code>" for c in canon) or "<i>none</i>"))
                if unmapped:
                    html.append("<br><span style='color:#c33'><b>Unmapped:</b> "
                                + ", ".join(f"<code>{u}</code>" for u in unmapped) + "</span>")
                html.append("<hr>")
            if not html:
                html.append("<i>Nothing to show for this scope.</i>")
            display(HTML("".join(html)))
            if any_unmapped:
                display(HTML("<p style='color:#c33'><b>❌ Some raw labels are still unmapped.</b> "
                             "Map them in section 3 and save again.</p>"))
            else:
                display(HTML("<p style='color:#178a17'><b>✅ All raw labels are mapped "
                             "(or explicitly ignored).</b></p>"))
        except Exception as e:
            display(HTML(f"<pre style='color:#c33'>{type(e).__name__}: {e}</pre>"))


# ========================= Wiring & layout =========================
chooser.register_callback(_update_info)
skip_existing.observe(lambda ch: render_harmonize(), names="value")
run_scan_button.on_click(run_scan)
preview_save_button.on_click(on_preview_save)
validate_button.on_click(on_validate)
verify_button.on_click(run_verify)

ui_layout = widgets.VBox([
    section1, chooser, csv_suffix, csv_suffix_info, existing_info, skip_existing, run_scan_button, out_scan,
    section2, out_configs,
    section3, out_harmonize, validate_button, out_validate,
    section4, widgets.HBox([fname_text, preview_save_button]), out_save,
    section5, widgets.HBox([verify_scope, verify_button]), out_verify,
])
display(ui_layout)
